In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
# from FEATURES.featuresV2 import *
# from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive
from math import log, exp
from scipy.stats import poisson
import numpy as np
from nba_api.stats.endpoints import leaguedashteamstats

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

In [ ]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

In [4]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]
dfs_data.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
244,Underdog,player_points,Jalen Brunson,Over,26.5,-137,2025-11-12,2025-11-12T00:17:54Z
245,Underdog,player_points,Jalen Brunson,Under,26.5,-137,2025-11-12,2025-11-12T00:17:54Z
246,Underdog,player_points,Karl-Anthony Towns,Over,21.5,-137,2025-11-12,2025-11-12T00:17:54Z
247,Underdog,player_points,Karl-Anthony Towns,Under,21.5,-137,2025-11-12,2025-11-12T00:17:54Z
248,Underdog,player_points,Ja Morant,Over,21.5,-137,2025-11-12,2025-11-12T00:17:54Z


# POINTS 

In [ ]:
res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_off_rtg = league_df['OFF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_ts = league_df['TS_PCT'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_pts = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_or = team_stats.at[player_team, 'OFF_RATING']
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']
    team_ts = team_stats.at[player_team, 'TS_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']

    # Opponent stats
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']

    df = s26
    try:
        player_df = df[df["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['PTS'].mean()
        else:
            lambda_base = player_df['PTS'].mean()
    except:
        lambda_base = player_df['PTS'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.93
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.04
    else:
        rest_factor = 0.98
    
    # Team offensive strength relative to league
    team_or_factor = cap_factor(team_or / league_avg_off_rtg)

    # Team assist-to-turnover ratio relative to league
    team_ast_ratio_factor = cap_factor(team_ast_ratio / league_avg_ast_ratio)
    
    # Team true shooting percentage relative to league
    team_ts_factor = cap_factor(team_ts / league_avg_ts)
    
    # Team offensive rebound percentage relative to league
    team_oreb_factor = cap_factor(team_oreb / league_avg_oreb)
    
    # Opponent defensive weakness relative to league (flip: lower def rating helps offense)
    opp_dr_factor = cap_factor(opp_dr / league_avg_def_rtg)
    
    # Pace adjustment relative to league
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (typical ~3% boost)
    home_factor = cap_factor(1.03 if home_flag else 0.97)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['PTS'].tail(7).mean()
    season_avg = player_df['PTS'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Usage rate adjustment (last 7 games vs season average)
    recent_usg_avg = player_df['USG_PCT'].tail(7).mean()
    season_usg_avg = player_df['USG_PCT'].mean()
    usg_factor = cap_factor(recent_usg_avg / season_usg_avg if season_usg_avg > 0 else 1.0)

    # Head-to-head adjustment (overall vs season average)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['PTS'].mean()
    else:
        h2h_avg = h2h['PTS'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_or_factor * 
                      opp_dr_factor * 
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      usg_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_pts % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_pts), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_pts) + 1)
    elif target_pts % 1 == 0:
        prob_over_poisson = poisson.sf(target_pts, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_pts), int(target_pts) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_pts,
        'LINE_TYPE': line_type,
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
point_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
point_df.head(10)

,NAME,LINE,LINE_TYPE,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Russell Westbrook,12.5,X.5 (need 13+),15.00,21.50,1.43,0.981,0.019,1.020
1,Andrew Wiggins,18.5,X.5 (need 19+),17.82,26.41,1.48,0.944,0.056,1.059
2,Cam Spencer,8.5,X.5 (need 9+),9.09,13.04,1.43,0.902,0.098,1.108
3,Tari Eason,11.5,X.5 (need 12+),11.44,16.03,1.40,0.875,0.125,1.143
4,Norman Powell,23.5,X.5 (need 24+),24.50,28.91,1.18,0.843,0.157,1.186
5,Terance Mann,9.5,X.5 (need 10+),9.80,12.64,1.29,0.809,0.191,1.236
6,Svi Mykhailiuk,8.5,X.5 (need 9+),9.30,11.43,1.23,0.804,0.196,1.244
7,Ziaire Williams,10.5,X.5 (need 11+),10.00,13.54,1.35,0.792,0.208,1.262
8,Landry Shamet,7.5,X.5 (need 8+),7.00,9.76,1.39,0.758,0.242,1.320
9,Josh Hart,9.5,X.5 (need 10+),8.25,11.97,1.45,0.755,0.245,1.325


# ASSISTS

In [16]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_assists')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_def_rtg = league_df['DEF_RATING'].mean()
league_avg_pace = league_df['PACE'].mean()
league_avg_ast_ratio = league_df['AST_RATIO'].mean()
league_avg_tov = league_df['TM_TOV_PCT'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_ast = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_ast_ratio = team_stats.at[player_team, 'AST_RATIO']

    # Opponent stats
    opp_dr = team_stats.at[opp_team_id, 'DEF_RATING']
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_tov = team_stats.at[opp_team_id, 'TM_TOV_PCT']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['AST'].mean()
        else:
            lambda_base = player_df['AST'].mean()
    except:
        lambda_base = player_df['AST'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.93
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.04
    else:
        rest_factor = 0.98
    
    # Team assist culture (pass-heavy teams create more assists)
    team_ast_ratio_factor = cap_factor(team_ast_ratio / league_avg_ast_ratio)
    
    # Opponent defensive weakness (weak defense = easier passes)
    opp_dr_factor = cap_factor(opp_dr / league_avg_def_rtg)
    
    # Opponent turnover pressure (teams that force turnovers limit assists)
    opp_tov_factor = cap_factor(league_avg_tov / opp_tov)
    
    # Pace adjustment (more possessions = more assist opportunities)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (smaller effect for assists)
    home_factor = cap_factor(1.02 if home_flag else 0.98)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['AST'].tail(7).mean()
    season_avg = player_df['AST'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Usage rate adjustment (last 7 games vs season average)
    recent_usg_avg = player_df['USG_PCT'].tail(7).mean()
    season_usg_avg = player_df['USG_PCT'].mean()
    usg_factor = cap_factor(recent_usg_avg / season_usg_avg if season_usg_avg > 0 else 1.0)

    # Head-to-head adjustment (assists vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['AST'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['AST'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_ast_ratio_factor * 
                      opp_dr_factor * 
                      opp_tov_factor *
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      usg_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_ast % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_ast), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_ast) + 1)
    elif target_ast % 1 == 0:
        prob_over_poisson = poisson.sf(target_ast, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_ast), int(target_ast) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_ast,
        'LINE_TYPE': line_type,
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
assist_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
assist_df.head(10)

,NAME,LINE,LINE_TYPE,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Nique Clifford,1.5,X.5 (need 2+),2.50,3.15,1.26,0.822,0.178,1.217
1,Christian Braun,2.5,X.5 (need 3+),3.11,4.15,1.33,0.783,0.217,1.278
2,Isaiah Collier,4.5,X.5 (need 5+),6.28,6.54,1.04,0.781,0.219,1.281
3,Jock Landale,1.5,X.5 (need 2+),1.64,2.58,1.57,0.728,0.272,1.374
4,Jamal Murray,5.5,X.5 (need 6+),5.62,7.23,1.29,0.728,0.272,1.374
5,Cedric Coward,2.5,X.5 (need 3+),2.91,3.49,1.20,0.678,0.322,1.475
6,Tari Eason,1.5,X.5 (need 2+),1.78,2.33,1.31,0.676,0.324,1.479
7,Ja Morant,7.5,X.5 (need 8+),7.90,8.81,1.11,0.653,0.347,1.531
8,Jerami Grant,2.5,X.5 (need 3+),2.40,2.89,1.20,0.552,0.448,1.812
9,Duncan Robinson,1.5,X.5 (need 2+),1.73,1.84,1.07,0.550,0.450,1.818


# REBOUNDS

In [17]:
dfs_data = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_rebounds')]

res = []
PLAYERS = (dfs_data[['NAME', 'LINE']].drop_duplicates(subset='NAME', keep='first'))

league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]

team_stats = league_df.set_index('TEAM_ID')
league_avg_pace = league_df['PACE'].mean()
league_avg_oreb = league_df['OREB_PCT'].mean()
league_avg_dreb = league_df['DREB_PCT'].mean()
league_avg_reb = league_df['REB_PCT'].mean()

def cap_factor(factor, min_val=0.85, max_val=1.15):
    """Cap adjustment factors to prevent extreme values"""
    return max(min_val, min(max_val, factor))

for _, row in PLAYERS.iterrows():
    PLAYER = row['NAME']
    target_reb = row['LINE']

    player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
    if player_df.empty:
        continue

    player_team = player_df['TEAM_ID'].iloc[-1]
    opp_team, home_flag = findOpp(PLAYER, player_df, current_date)

    opp_matches = s26[s26['TEAM_ABBREVIATION'] == opp_team]
    if opp_matches.empty:
        continue
    opp_team_id = opp_matches['TEAM_ID'].iloc[-1]

    # Team stats
    team_pace = team_stats.at[player_team, 'PACE']
    team_reb = team_stats.at[player_team, 'REB_PCT']
    team_oreb = team_stats.at[player_team, 'OREB_PCT']
    team_dreb = team_stats.at[player_team, 'DREB_PCT']

    # Opponent stats
    opp_pace = team_stats.at[opp_team_id, 'PACE']
    opp_reb = team_stats.at[opp_team_id, 'REB_PCT']
    opp_oreb = team_stats.at[opp_team_id, 'OREB_PCT']
    opp_dreb = team_stats.at[opp_team_id, 'DREB_PCT']

    # Baseline lambda
    try:
        player_df = s26[s26["PLAYER_NAME"] == PLAYER].copy()
        player_df_25 = s25[s25["PLAYER_NAME"] == PLAYER].copy()
        if len(player_df) < 5:  
            lambda_base = player_df_25['REB'].mean()
        else:
            lambda_base = player_df['REB'].mean()
    except:
        lambda_base = player_df['REB'].mean()

    if lambda_base <= 0:
        print(f"Skipping {PLAYER} - invalid baseline lambda")
        continue
    
    # Days Rested
    current_date_dt = pd.to_datetime(current_date)
    player_df['GAME_DATE'] = pd.to_datetime(player_df['GAME_DATE'])

    last_game_date = player_df['GAME_DATE'].max()
    days_rested = (current_date_dt - last_game_date).days 
    if days_rested == 0:
        rest_factor = 0.95  # Slightly less penalty than scoring
    elif days_rested == 1:
        rest_factor = 1.00
    elif days_rested == 2:
        rest_factor = 1.02
    elif days_rested >= 3 and days_rested <= 5:
        rest_factor = 1.03
    else:
        rest_factor = 0.99
    
    # Team rebounding culture (rebounding-focused teams)
    team_reb_factor = cap_factor(team_reb / league_avg_reb)
    
    # Opponent gives up offensive rebounds (weak DREB% = more offensive boards available)
    opp_dreb_factor = cap_factor(league_avg_dreb / opp_dreb)
    
    # Opponent gives up defensive rebounds (weak OREB% = more defensive boards available)
    opp_oreb_factor = cap_factor(league_avg_oreb / opp_oreb)
    
    # Pace adjustment (more possessions = more missed shots = more rebounds)
    expected_pace = (team_pace + opp_pace) / 2
    pace_factor = cap_factor(expected_pace / league_avg_pace)

    # Home court advantage (minimal effect for rebounds)
    home_factor = cap_factor(1.01 if home_flag else 0.99)

    # Recent form adjustment (last 7 games vs season average)
    recent_avg = player_df['REB'].tail(7).mean()
    season_avg = player_df['REB'].mean()
    form_factor = cap_factor(recent_avg / season_avg if season_avg > 0 else 1.0)

    # Minutes adjustment (last 7 games vs season average)
    recent_min_avg = player_df['MIN'].tail(7).mean()
    season_min_avg = player_df['MIN'].mean()
    min_factor = cap_factor(recent_min_avg / season_min_avg if season_min_avg > 0 else 1.0)

    # Head-to-head adjustment (rebounds vs this opponent)
    h2h = player_df[player_df['OPP_ABBREVIATION'] == opp_team]
    if h2h.empty:
        player_df_25 = s25[s25['OPP_ABBREVIATION'] == opp_team]
        h2h_avg = player_df_25['REB'].mean() if not player_df_25.empty else season_avg
    else:
        h2h_avg = h2h['REB'].mean()
    h2h_factor = cap_factor(h2h_avg / season_avg if season_avg > 0 else 1.0)

    # Combine all factors
    combined_factor = (team_reb_factor * 
                      opp_dreb_factor * 
                      opp_oreb_factor *
                      pace_factor * 
                      home_factor * 
                      form_factor * 
                      min_factor * 
                      rest_factor * 
                      h2h_factor)

    # Adjust lambda
    lambda_adjusted = lambda_base * combined_factor

    # Determine which Poisson calculation to use based on line type
    if target_reb % 1 == 0.5:
        prob_over_poisson = poisson.sf(int(target_reb), lambda_adjusted)
        line_type = "X.5 (need {}+)".format(int(target_reb) + 1)
    elif target_reb % 1 == 0:
        prob_over_poisson = poisson.sf(target_reb, lambda_adjusted)
        line_type = "Over {} (need {}+)".format(int(target_reb), int(target_reb) + 1)

    res.append({
        'NAME': PLAYER,
        'LINE': target_reb,
        'LINE_TYPE': line_type,
        'BASELINE_LAMBDA': round(lambda_base, 2),
        'ADJUSTED_LAMBDA': round(lambda_adjusted, 2),
        'COMBINED_FACTOR': round(combined_factor, 2),
        'OVER%': round(prob_over_poisson, 3),
        'UNDER%': round(1 - prob_over_poisson, 3),
        'IMPLIED_ODDS': round(1 / prob_over_poisson, 3) if prob_over_poisson > 0 else None,
    })
    
rebound_df = pd.DataFrame(res).sort_values(by='OVER%', ascending=False).reset_index(drop=True)
rebound_df.head(10)

,NAME,LINE,LINE_TYPE,BASELINE_LAMBDA,ADJUSTED_LAMBDA,COMBINED_FACTOR,OVER%,UNDER%,IMPLIED_ODDS
0,Jamal Shead,1.5,X.5 (need 2+),1.90,2.73,1.44,0.757,0.243,1.320
1,Moses Moody,2.5,X.5 (need 3+),2.89,3.95,1.37,0.755,0.245,1.325
2,Jaylin Williams,4.5,X.5 (need 5+),4.45,5.84,1.31,0.693,0.307,1.443
3,Domantas Sabonis,12.5,X.5 (need 13+),14.00,14.33,1.02,0.673,0.327,1.485
4,Keyonte George,3.5,X.5 (need 4+),3.80,4.49,1.18,0.656,0.344,1.524
5,Ajay Mitchell,3.5,X.5 (need 4+),4.00,4.30,1.07,0.623,0.377,1.606
6,Christian Braun,4.5,X.5 (need 5+),4.78,5.28,1.10,0.607,0.393,1.649
7,Jalen Duren,13.5,X.5 (need 14+),12.00,14.69,1.22,0.606,0.394,1.649
8,Isaiah Hartenstein,10.5,X.5 (need 11+),11.82,11.15,0.94,0.558,0.442,1.792
9,Duncan Robinson,2.5,X.5 (need 3+),2.64,2.90,1.10,0.554,0.446,1.806
